In [65]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/BTT Fall AI Studio (swytch 1b)'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
import os
os.makedirs(f'{BASE}/data/raw', exist_ok=True)
os.makedirs(f'{BASE}/data/interim', exist_ok=True)
os.makedirs(f'{BASE}/notebooks', exist_ok=True)

In [67]:
import pandas as pd

RAW_PATH = f'{BASE}/data/raw/occ_level.csv'
OUT_PATH = f'{BASE}/eloundou_soc6.csv'

RATING_COLS = [
    "dv_rating_alpha", "dv_rating_beta", "dv_rating_gamma",
    "human_rating_alpha", "human_rating_beta", "human_rating_gamma",
]

In [68]:
df = pd.read_csv(RAW_PATH)
assert df.shape[0] == 923, f"Expected 923 rows, got {df.shape[0]}"

df['soc6'] = df['O*NET-SOC Code'].str.split('.').str[0]

rolled = (
    df.groupby('soc6')
    .agg(
        **{col: (col, 'mean') for col in RATING_COLS},
        n_children=('O*NET-SOC Code', 'count'),
        child_codes=('O*NET-SOC Code', lambda x: ';'.join(x)),
        child_titles=('Title', lambda x: '; '.join(x)),
    )
    .reset_index()
)

print(f"Rows in: {len(df)} | Rows out: {len(rolled)}")
print(f"Codes rolled up from >1 child: {(rolled['n_children'] > 1).sum()}")
rolled.head()

Rows in: 923 | Rows out: 798
Codes rolled up from >1 child: 67


,soc6,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma,n_children,child_codes,child_titles
0,11-1011,0.133333,0.507778,0.882222,0.117778,0.369444,0.621111,2,11-1011.00;11-1011.03,Chief Executives; Chief Sustainability Officers
1,11-1021,0.000000,0.480769,0.961538,0.115385,0.384615,0.653846,1,11-1021.00,General and Operations Managers
2,11-1031,0.033333,0.400000,0.766667,0.266667,0.516667,0.766667,1,11-1031.00,Legislators
3,11-2011,0.000000,0.476744,0.953488,0.255814,0.546512,0.837209,1,11-2011.00,Advertising and Promotions Managers
4,11-2021,0.062500,0.500000,0.937500,0.218750,0.578125,0.937500,1,11-2021.00,Marketing Managers


In [69]:
OUT_PATH = f'{BASE}/data/interim/eloundou_soc6.csv'
rolled.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

Saved to /content/drive/MyDrive/BTT Fall AI Studio (swytch 1b)/data/interim/eloundou_soc6.csv


In [70]:
AIOE_PATH = f'{BASE}/data/raw/AIOE_DataAppendix.xlsx'
CROSSWALK_PATH = f'{BASE}/data/raw/soc_2010_to_2018_crosswalk.xlsx'
OUT_PATH = f'{BASE}/data/interim/aioe_soc2018.csv'
COVERAGE_OUT_PATH = f'{BASE}/data/interim/aioe_coverage_log.csv'

In [71]:
aioe = pd.read_excel(AIOE_PATH, sheet_name='Appendix A')
assert aioe.shape[0] == 774, f"Expected 774 AIOE rows, got {aioe.shape[0]}"
aioe = aioe.rename(columns={'SOC Code': 'soc2010', 'Occupation Title': 'title_2010'})

cw = pd.read_excel(CROSSWALK_PATH, sheet_name='Sorted by 2010', skiprows=8)
cw = cw.rename(columns={
    '2010 SOC Code': 'soc2010', '2010 SOC Title': 'title_2010_cw',
    '2018 SOC Code': 'soc2018', '2018 SOC Title': 'title_2018',
})

In [72]:
merged = aioe.merge(cw, on='soc2010', how='left', indicator=True)

no_match = merged[merged['_merge'] == 'left_only'].copy()
no_match['reason'] = 'no_crosswalk_match'

matched = merged[merged['_merge'] == 'both'].copy().drop(columns='_merge')
matched['is_split'] = matched.groupby('soc2010')['soc2018'].transform('nunique') > 1
matched['is_merge'] = matched.groupby('soc2018')['soc2010'].transform('nunique') > 1

rolled = (
    matched.groupby('soc2018')
    .agg(
        AIOE=('AIOE', 'mean'),
        title_2018=('title_2018', 'first'),
        n_2010_parents=('soc2010', 'nunique'),
        parent_soc2010_codes=('soc2010', lambda x: ';'.join(sorted(set(x)))),
        is_split=('is_split', 'any'),
        is_merge=('is_merge', 'any'),
    )
    .reset_index()
)

print(f"Rows out: {len(rolled)} | No match: {len(no_match)} | Split: {rolled['is_split'].sum()} | Merge: {rolled['is_merge'].sum()}")
rolled.head()

Rows out: 800 | No match: 1 | Split: 76 | Merge: 24


,soc2018,AIOE,title_2018,n_2010_parents,parent_soc2010_codes,is_split,is_merge
0,11-1011,1.334246,Chief Executives,1,11-1011,False,False
1,11-1021,0.574877,General and Operations Managers,1,11-1021,False,False
2,11-2011,1.294387,Advertising and Promotions Managers,1,11-2011,False,False
3,11-2021,1.315032,Marketing Managers,1,11-2021,False,False
4,11-2022,1.266280,Sales Managers,1,11-2022,False,False


In [73]:
rolled.to_csv(OUT_PATH, index=False)
no_match.to_csv(COVERAGE_OUT_PATH, index=False)

In [74]:
!pip install pdfplumber -q

In [75]:
import re
import pdfplumber
import pandas as pd

PDF_PATH = f'{BASE}/data/raw/The_Future_of_Employment.pdf'
CROSSWALK_PATH = f'{BASE}/data/raw/soc_2010_to_2018_crosswalk.xlsx'
APPENDIX_START_PAGE = 56

ROW_PATTERN = re.compile(r"^(\d+)\.\s+([\d.]+)\s+(?:(\d)\s+)?(\d{2}-\d{4})\s+(.+)$")

rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for i in range(APPENDIX_START_PAGE, len(pdf.pages)):
        text = pdf.pages[i].extract_text() or ""
        for line in text.split("\n"):
            m = ROW_PATTERN.match(line.strip())
            if m:
                rank, prob, label, soc, title = m.groups()
                rows.append({"rank": int(rank), "probability": float(prob),
                             "training_label": label, "soc2010": soc, "title_2010": title})

fo = pd.DataFrame(rows)
assert fo.shape[0] == 702, f"Expected 702 rows, got {fo.shape[0]}"
fo.head()

,rank,probability,training_label,soc2010,title_2010
0,1,0.0028,None,29-1125,RecreationalTherapists
1,2,0.0030,None,49-1011,"First-LineSupervisorsofMechanics,Installers,an..."
2,3,0.0030,None,11-9161,EmergencyManagementDirectors
3,4,0.0031,None,21-1023,MentalHealthandSubstanceAbuseSocialWorkers
4,5,0.0033,None,29-1181,Audiologists


In [76]:
cw = pd.read_excel(CROSSWALK_PATH, sheet_name='Sorted by 2010', skiprows=8)
cw = cw.rename(columns={'2010 SOC Code': 'soc2010', '2010 SOC Title': 'title_2010_cw',
                          '2018 SOC Code': 'soc2018', '2018 SOC Title': 'title_2018'})

merged = fo.merge(cw, on='soc2010', how='left', indicator=True)
no_match = merged[merged['_merge'] == 'left_only'].copy()
no_match['reason'] = 'no_crosswalk_match'

matched = merged[merged['_merge'] == 'both'].copy().drop(columns='_merge')
matched['is_split'] = matched.groupby('soc2010')['soc2018'].transform('nunique') > 1
matched['is_merge'] = matched.groupby('soc2018')['soc2010'].transform('nunique') > 1

rolled = (
    matched.groupby('soc2018')
    .agg(probability=('probability', 'mean'), title_2018=('title_2018', 'first'),
         n_2010_parents=('soc2010', 'nunique'),
         parent_soc2010_codes=('soc2010', lambda x: ';'.join(sorted(set(x)))),
         is_split=('is_split', 'any'), is_merge=('is_merge', 'any'))
    .reset_index()
)

print(f"Rows out: {len(rolled)} | No match: {len(no_match)} | Split: {rolled['is_split'].sum()} | Merge: {rolled['is_merge'].sum()}")

Rows out: 699 | No match: 17 | Split: 48 | Merge: 16


In [77]:
rolled.to_csv(f'{BASE}/data/interim/frey_osborne_soc2018.csv', index=False)
no_match.to_csv(f'{BASE}/data/interim/frey_osborne_coverage_log.csv', index=False)

In [78]:
elo = pd.read_csv(f'{BASE}/data/interim/eloundou_soc6.csv').rename(columns={'soc6': 'soc'})
aioe = pd.read_csv(f'{BASE}/data/interim/aioe_soc2018.csv').rename(columns={
    'soc2018': 'soc', 'title_2018': 'title_aioe',
    'n_2010_parents': 'aioe_n_2010_parents', 'parent_soc2010_codes': 'aioe_parent_soc2010_codes',
    'is_split': 'aioe_is_split', 'is_merge': 'aioe_is_merge'})
fo = pd.read_csv(f'{BASE}/data/interim/frey_osborne_soc2018.csv').rename(columns={
    'soc2018': 'soc', 'title_2018': 'title_frey_osborne',
    'n_2010_parents': 'fo_n_2010_parents', 'parent_soc2010_codes': 'fo_parent_soc2010_codes',
    'is_split': 'fo_is_split', 'is_merge': 'fo_is_merge'})

m = elo.merge(aioe, on='soc', how='outer', indicator='in_aioe')
m = m.merge(fo, on='soc', how='outer', indicator='in_fo')

m['has_eloundou'] = m['dv_rating_alpha'].notna()
m['has_aioe'] = m['AIOE'].notna()
m['has_frey_osborne'] = m['probability'].notna()
m['n_sources'] = m[['has_eloundou', 'has_aioe', 'has_frey_osborne']].sum(axis=1)

final = m[m['n_sources'] == 3].copy().drop(columns=['in_aioe', 'in_fo'])
dropped = m[m['n_sources'] < 3].copy()

def reason(row):
    missing = [s for s, has in [('eloundou', row['has_eloundou']),
                                  ('aioe', row['has_aioe']),
                                  ('frey_osborne', row['has_frey_osborne'])] if not has]
    return 'missing_from_' + '_and_'.join(missing)

dropped['reason'] = dropped.apply(reason, axis=1)
dropped = dropped[['soc', 'has_eloundou', 'has_aioe', 'has_frey_osborne', 'reason']]

print(f"Final table: {len(final)} | Dropped: {len(dropped)}")
print(dropped['reason'].value_counts())

import os
os.makedirs(f'{BASE}/data/processed', exist_ok=True)
final.to_csv(f'{BASE}/data/processed/merged_ai_risk_v0.csv', index=False)
dropped.to_csv(f'{BASE}/data/processed/coverage_report_v0.csv', index=False)

Final table: 693 | Dropped: 115
reason
missing_from_frey_osborne                 97
missing_from_aioe_and_frey_osborne         8
missing_from_eloundou                      6
missing_from_eloundou_and_frey_osborne     4
Name: count, dtype: int64


In [79]:
print(final[['AIOE', 'probability']].corr(method='spearman'))

                 AIOE  probability
AIOE         1.000000    -0.363438
probability -0.363438     1.000000
